# Pandas

* Jake VanderPlas. 2016. *Python Data Science Handbook: Essential Tools for Working with Data*. O'Reilly Media, Inc.
* 3. kapitulua - Datuen manipulazioa Pandas-ekin
* https://github.com/jakevdp/PythonDataScienceHandbook

Pandas-ek honako hauek eskaintzen ditu:

* Sarrera/irteerako gaitasun aberatsak (CSV, Excel, SQL, JSON eta abarretatik datuak irakurri eta horietan idaztea)
* Dimentsio bateko (**Series**) eta bi dimentsioko taula-formako (**DataFrame**) datu-egiturak.
* Datuen malgutasuna (falta diren datuak, denbora-serieak eta datu mota heterogeneoak kudeatzen ditu).
* Etiketatutako errenkadak eta zutabeak, datuak lerrokatzeko
* Indexazio malgua, *slicing*-a, *fancy indexing*-a eta datu multzo handietarako azpimultzo-hautaketa .

In [ ]:
import numpy as np
import pandas as pd

pd.__version__

In [ ]:
# sakatu TAB Pandas-en izen-espazioa ikusteko
#pd.

In [ ]:
def pprint(*args, sep='\n', end='\n', userepr=True, align=False, breakline=False, indent='   '):
    "Ebaluatu eta modu txukunean inprimatu"
    if align :
        max_arg_len = max(map(len,args))
        txt2txt = lambda txt : ' '*(max_arg_len-len(txt)) + txt
        args = list(map(txt2txt, args))
    geval = lambda txt : eval(txt, globals())
    txt2txt = lambda txt : f'{repr(geval(txt))}' if userepr else f'{geval(txt)}'    
    output = map(txt2txt, args)
    txt2txt = lambda txt : f'{txt} = '    
    prefix = map(txt2txt, args)
    if breakline :
        po2txt = lambda p,o : (p+'\n'+o).replace('\n','\n'+indent)
    else:
        po2txt = lambda p,o : (p+o).replace('\n','\n'+' '*len(p))
    print(*map(po2txt, prefix, output), sep=sep, end=end)

class display(object):
    """Bistaratu hainbat objekturen HTML adierazpena"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)

from IPython.display import HTML
HTML("""
<style>
.dataframe {
    font-size: 80% !important; /* Adjust the percentage as needed */
}
</style>
""")

## Pandas Series-ak

* Indexatutako dimentsio bakarreko arraya
* Bi atributu nagusi:
   * `values` : NumPy arraya
   * `index` : `pd.Index` motako array erako objektu bat 

In [ ]:
d = pd.Series([0.25, 0.5, 0.75, 1.0])
pprint('d', 'd.values', 'd.index', align=True)

Series-ak NumPy arrayetatik sor daitezke:

In [ ]:
d = pd.Series(np.linspace(0,4,6))
d

Series bat NumPy array bat bezala indexa daiteke:

In [ ]:
d = pd.Series(np.arange(10,15))
pprint('d[3]')  # indize sinplea --> eskalarra
pprint('d[3:]', 'd[3:4]')  # zatia --> Series-a
pprint('d[[1,3,0]]', 'd[[1]]')  # indize aurreratua --> Series-a

Baina... Pandas Series-ek **indize esplizitu** bat izan dezakete
   * Ematen ez bada, **osoko indize inplizitu** bat erabiltzen da `[0,1,...,n-1]`

In [ ]:
d = pd.Series(np.arange(10,15), index=["a","b","c","d","e"])
pprint('d')
# ABISUA "treating keys as positions is deprecated"  --> use Series.iloc[pos]
#pprint('d[1]')
pprint('d["b"]')

### Slicing - 1. kasua - etiketetak indize esplizituan
   * Slicean zenbaki osoak ez diren (esplizituak) nahiz osoak diren (inplizituak) balioak erabil daitezke.
   * slicea ez-osoa &rarr; **etiketetan oinarritutako indexazioa** &rarr; $[from,to]$
   * slicea osoa &rarr; **posizioan oinarritutako indexazioa** &rarr; $[from,to)$

<br>

In [ ]:
d = pd.Series([100,101,102,103,104], index=["a","b","c","d","e"])
pprint('d')
pprint('d["a"]', 'd["c"]', 'd["a":"c"].values')   # indize esplizitua
pprint('d[0:2].values')                           # indize inplizitua

### Slicing - 2. kasua - zenbaki osoak indize esplizituan
   * Slicing-a <u>beti indize inplizituei dagokie</u>.
   * slicea osoa &rarr; **posizioan oinarritutako indexazioa** &rarr; $[from,to)$

<br>

In [ ]:
d = pd.Series([100,101,102,103,104], index=[1,5,4,2,3])
pprint('d')
pprint('d[1]', 'd[3]')  # indize esplizitua
pprint('d[1:3].values') # indize inplizitua

### Fancy Indexing
   * Etiketetan oinarritutako indexazioa
   * Posizioan oinarritutako indexazioa zaharkituta dago 

<br>

In [ ]:
d = pd.Series([100,101,102,103,104], index=["a","b","c","d","e"])
pprint('d[["a","b","c"]].values')   # indize esplizitua
# ABISUA "treating keys as positions is deprecated"  --> use Series.iloc[pos]
#pprint('d[[1,2,3]].values')   # indize inplizitua

d = pd.Series([100,101,102,103,104], index=[1,5,4,2,3])
pprint('d[[1,2,3]].values')   # indize esplizitua

### Series-ak eta hiztegiak
* Series-ak Python-eko hiztegi espezializatu modukoak dira $\{index_{typed \,\&\, ordered} \to value_{typed}\}$ 
* `pd.Series(dict)` &rarr; hiztegi batetik Series bat sortu
* `pd.Series.to_dict()` &rarr; Series batetik hiztegi bat sortu<br><br>

In [ ]:
d1 = pd.Series([100,101,102], index=["a","b","c"])
x = d1.to_dict()
d2 = pd.Series(x)
pprint('d1','x','d2')

* `.keys()` &rarr; `.index` <br><br>

In [ ]:
d = pd.Series([100,101,102,103,104], index=["a","b","c","d","e"])
pprint('type(d.index) == type(d.keys())')
pprint('d.index == d.keys()')
pprint('d.index is d.keys()')

 <br>
 
 * $\nexists\;\;$ `.values()`

In [ ]:
d.values
# TypeError: 'numpy.ndarray' object is not callable
#d.values()

* `.items()` &rarr; `zip` objektua: `(index,value)` <br><br>

In [ ]:
d = pd.Series([100,101,102,103,104], index=["a","b","c","d","e"])
pprint('d.items()')
print(*d.items())


<br>

* `x in series` &rarr; Series-aren indizean `x` ote dagoen egiaztatzen 

In [ ]:
d = pd.Series([100,101,102,103,104], index=["a","b","c","d","e"])
pprint('102 in d', '"xxx" in d')

## Pandas DataFrame-ak

* DataFrame-ak Python-eko hiztegi espezializatu modukoak dira $\{index_{typed \,\&\, ordered} \to series\}$ errenkada-indize komun batekin
* Errenkada-indize komuneko *Series-en Series* moduko bat
* * Hiru atributu nagusi:
   * `values` : NumPy arraya
   * `index` eta `columns`: `pd.Index` motako array gisako objektuak 

<br>

In [ ]:
cities = ['California','Texas','Florida','New York']
population = pd.Series([39538223,29145505,21538187,20201249], index=cities)
area = pd.Series([423967, 695662, 170312, 141297], index=cities)
states = pd.DataFrame({'population':population, 'area':area})
states

* `index` atributua (*errenkada-indizea*) eta `columns` atributua (*zutabe-indizea*) `pd.Index` motakoak dira
* `values` atributuak (`np.ndarray` motakoa) datuak dauzka

<br>

In [ ]:
states.index

<br>

In [ ]:
states.columns

<br>

In [ ]:
states.values

DataFrame bat Series bakar batetik eraiki daiteke:<br><br>

In [ ]:
cities = ['California','Texas','Florida','New York']
population = pd.Series([39538223,29145505,21538187,20201249], index=cities)
states = pd.DataFrame(population, columns=['population'])
states

DataFrame bat Series hiztegi batetik eraiki daiteke:<br><br>

In [ ]:
cities = ['California','Texas','Florida','New York']
population = pd.Series([39538223,29145505,21538187,20201249], index=cities)
area = pd.Series([423967, 695662, 170312, 141297], index=cities)
states = pd.DataFrame({'population':population, 'area':area})
states

DataFrame bat hiztegi zerrenda batetik eraiki daiteke:<br><br>

In [ ]:
data = [
    {'population':39538223, 'area':423967},
    {'population':29145505, 'area':695662},
    {'population':21538187, 'area':170312},
    {'population':20201249, 'area':141297}
]
states = pd.DataFrame(data, index=["California","Texas","Florida","New York"])
states

DataFrame bat bi dimentsioko NumPy array batetik eraiki daiteke:<br><br>

In [ ]:
data = np.array([[39538223,   423967],
          [29145505,   695662],
          [21538187,   170312],
          [20201249,   141297]])
states = pd.DataFrame(data,
                      columns=["population","area"],
                      index=["California","Texas","Florida","New York"])
states

DataFrame batek orden desberdineko indizeak integra ditzake<br><br>

In [ ]:
cities1 = ['California','Texas','Florida','New York']
population = pd.Series([39538223,29145505,21538187,20201249], index=cities1)
cities2 = ['Texas','Florida','New York','California']
area = pd.Series([695662, 170312, 141297, 423967], index=cities2)
states = pd.DataFrame({'population':population, 'area':area})
states

DataFrame batek balio desberdineko indizeak integra ditzake
* *Falta diren* balioak `NaN` balioekin betetzen dira <br><br>

In [ ]:
cities1 = ['California','Texas','Florida']
population = pd.Series([39538223,29145505,21538187], index=cities1)
cities2 = ['Texas','Florida','New York']
area = pd.Series([695662, 170312, 141297], index=cities2)
states = pd.DataFrame({'population':population, 'area':area})
states

## Pandas Index-a
* NumPy-ko array aldaezin moduko bat
* NumPy-ko arrayen atributu asko ditu; indexatu, slicing eta abar egin daiteke. <br><br>

In [ ]:
ind = pd.Index(['California', 'Texas', 'Florida'])
pprint('ind', 'ind[1]', 'ind[:2]', 'ind[[2,0,1]]', align=True, end='\n\n')
pprint('ind.size', 'ind.shape', 'ind.ndim', 'ind.dtype', align=True)

In [ ]:
ind = pd.Index([200, 1000, 300])
pprint('ind', 'ind[1]', 'ind[:2]', 'ind[[2,0,1]]', align=True, end='\n\n')
pprint('ind.size', 'ind.shape', 'ind.ndim', 'ind.dtype', align=True)

`pd.Index`-k Python-eko `set` objektuen konbentzio asko jarraitzen ditu: 

* `idx1.union(idx2)` &rarr; bi Index-etako elementuak konbinatzen ditu
* `idx1.ebakidura(idx2)` &rarr; bi Index-etan komunak diren elementuak
* `idx1.difference(idx2)` &rarr; `idx1`-en egon eta `idx2`-n ez dauden elementuak
* `idx1.symmetric_difference(idx2)` &rarr; soilik bietako batean dauden elementuak
* `idx1.equals(idx2)` &rarr; elementu berak eta orden berean dituzten egiaztatzen du

<br>

In [ ]:
idx1 = pd.Index([1,2,3,4,5])
idx2 = pd.Index([4,5,6,7])
pprint('idx1.union(idx2)', 'idx1.intersection(idx2)',
       'idx1.difference(idx2)', 'idx1.symmetric_difference(idx2)',
       'idx1.equals(idx2)', align=True)

## Pandas-en Sarrera/Irteera
* S/I funtzio-bilduma ahaltsu eta malgua
   * Testu-fitxategiak: CSV, TSV, testu arrunta
   * Kalkulu-orriak: Excel, OpenOffice/LibreOffice
   * Webeko datuak: HTML
   * Datu-formatu egituratuak: JSON, XML
   * Datu-base erlazionalak: SQL
   * Beste formatu batzuk: Pickle, HDF5, Parquet, ...
* Sarrera (`pd.read_...`): `pd.read_csv`, `pd.read_excel`, `pd.read_json`,... `
* Irteera (`df.to_...`): `df.to_csv`, `df.to_excel`, `df.to_json`,... `


In [ ]:
#pd.read_
#df.to_

In [ ]:
url = "https://raw.githubusercontent.com/pandas-dev/pandas/refs/heads/main/pandas/tests/io/data/csv/iris.csv"
pd.read_csv(url)

In [ ]:
#!pip install openpyxl
url = "https://github.com/pandas-dev/pandas/raw/refs/heads/main/pandas/tests/io/data/excel/test1.xlsx"
pd.read_excel(url, index_col=0) # erabili lehen zutabea indize gisa

## Indexazioa eta hautaketa

In [ ]:
cities = ['California', 'Texas', 'Florida', 'New York', 'Pennsylvania']
area = pd.Series([423967, 695662, 170312, 141297, 119280], index=cities)
population = pd.Series([39538223, 29145505, 21538187, 20201249, 13002700], index=cities)
gdp = pd.Series([3.9, 2.7, 1.3, 1.8, 1.0], index=cities) # Barne Produktu Gordina 
df = pd.DataFrame({'area':area, 'population':population, 'GDP':gdp})
df

* Indexazio sinplea &rarr; zutabe-hautaketa (**Series**)

In [ ]:
df['area']
# KeyError: 0
#df[0]

* Fancy Indexing &rarr; zutabeen hautaketa (**DataFrame**)

In [ ]:
display( "df[['GDP','area']]", "df[['area']]")

In [ ]:
pprint("df['GDP']")

* Slicing &rarr; errenkaden hautaketa (**DataFrame**)

In [ ]:
display('df[:2]', 'df[1:2]', 'df[2:2]')

### Zutabe eta errenkaden hautaketen konbinaketa
* zutabeak fancy indexing-arekin hautatzen dira
* errenkadak (posizioaren araberako) slicing-arekin hautatzen dira

In [ ]:
display("df[['area','population']][:3]" , "df[:3][['area','population']]")

### Datuen atzipena `df.values` bidez 
   * `df.values` &rarr; NumPy array homogeneoa
   * Series-en datu motak goragoko mota batera bihur daitezke (*upcasting*)

In [ ]:
pprint('df.values')
pprint('df.values.dtype', 'df["area"].dtype',
       'df["population"].dtype', 'df["GDP"].dtype', align=True)

### Konparazio-eragilearekin maskaratzea
* Izan bedi `op` eragile bat
* `Series op value` &rarr; broadcasting eragiketa &rarr; Series
* `DataFrame op value` &rarr; broadcasting eragiketa &rarr; DataFrame<br><br>

In [ ]:
pprint("df['GDP'] > 2")

In [ ]:
display("df * 10" , "df > 2")

Series/DataFrame boolearrak indize gisa erabil daitezke
* Series boolearra &rarr; errenkadak hautatu
* DataFrame boolearra &rarr; elementuak hautatu (eta `NaN` balioekin bete)

In [ ]:
display("df[df['GDP'] > 2]" , "df[df > 2]")

`GDP>1` eta `population<30000000` betetzen duten hirien azalera: :

In [ ]:
df[df['GDP'] > 1]

In [ ]:
df[(df['GDP']>1) & (df['population']<30000000)]

In [ ]:
df[(df['GDP']>1) & (df['population']<30000000)]['area']

### Indexatzaileak: `.loc` and `.iloc`
* `.loc` &rarr; indexazio (eta slicing) esplizitua 
* `.iloc` &rarr; indexazio (eta slicing) inplizitua
* NumPy moduko indexazioa
   * `.loc[i]` &rarr; `i` errenkada 
   * `.loc[i,j]` &rarr; `i` errenkada, `j` zutabea 
   * Indexazio sinplea, fancy indexing and slicing

In [ ]:
df

In [ ]:
df.loc['Florida']

In [ ]:
df.loc[['Florida','Texas']]

In [ ]:
df.loc[:'Florida']

In [ ]:
df.iloc[2]

In [ ]:
df.iloc[[2,1]]

In [ ]:
df.iloc[:3]

In [ ]:
df.loc['Florida','population']

In [ ]:
df.loc[['California','Florida'],'population']

In [ ]:
df.loc[['California','Florida'],:]

`.iloc` erabiltzea `df.values` erabiltzearen antzekoa da, baina DataFrame-aren egitura mantentzen du (NumPy array homogeneoaren aldean)

In [ ]:
df.iloc[:2,[0,2]]

In [ ]:
pprint("df.values[:2,[0,2]]" , "df.values[:2,[0,2]].dtype", align=True)

## DataFrame-ak indexazio bidez aldatzen
* Indexazioa DataFrame bat aldatzeko erabil daiteke
   * Balioak aldatu
   * Zutabeak gehitu/kendu
   * Errenkadak gehitu/kendu

### DataFrame-eko balioak aldatzen

* Indexazio sinplea &rarr; zutabe-esleipena

In [ ]:
df2 = df.copy()
df2['area'] = 0
df2

* Fancy Indexing &rarr; zutabeen esleipena

In [ ]:
df2 = df.copy()
df2[['area','GDP']] = 0
df2

* Slicing &rarr; errenkaden esleipena

In [ ]:
df2 = df.copy()
df2[:2] = 0
df2

* `df.values[i,j]`-ri esleitzeak ez du DataFrame-a aldatzen

In [ ]:
df2 = df.copy()
df2.values[:2,:2] = 0
df2

* Series bidezko maskaratzea (indizea Series boolear bat da) &rarr; errenkaden esleipena

In [ ]:
df2 = df.copy()
pprint('df2["GDP"]>2')
df2[df2['GDP']>2] = 0
df2

* DataFrame bidezko maskaratzea (indizea DataFrame boolear bat da) &rarr; *elementuen* esleipena

In [ ]:
df2 = df.copy()
pprint('df2>2')
df2[df2>2] = 0
df2

* `.loc` and `.iloc` &rarr; *azpimultzoen* esleipena

In [ ]:
df2 = df.copy()
df2.loc[['California','New York'],'population':] = 0
df2

In [ ]:
df2 = df.copy()
df2.iloc[[0,3],1:] = 0
df2

### Zutabeak gehitzen/kentzen
   * **Gehitu**: `df[new_index] = value`
   * **Kendu**: `del df[index]` edo `df.drop(index, axis=1, inplace=True)`

In [ ]:
# gehitu 'density' zutabea 
df2 = df.copy()
df2['density'] = df2['population'] / df2['area']
df2

In [ ]:
# kendu zutabea 'population'
df2 = df.copy()
del df2['population']
df2

In [ ]:
# kendu 'population' zutabea
df2 = df.copy()
df2.drop('population', axis=1, inplace=True)
df2

In [ ]:
# kendu ['area','population'] zutabeak
df2 = df.copy()
df2.drop(['area','population'], axis=1, inplace=True)
df2

### Errenkadak gehitzen/kentzen
   * **Gehitu**: `df.loc[new_index] = value` (WARNING &rarr; homogeneous DataFrame)
   * **Kendu**: `df.drop(index, axis=0, inplace=True)`

In [ ]:
# gehitu 'chicago' errenkada
df2 = df.copy()
df2.loc['Chicago'] = [111111,22222222,2.0]
df2

In [ ]:
# kendu 'New York' errenkada
df2 = df.copy()
df2.drop('New York', axis=0, inplace=True)
df2

In [ ]:
# kendu ['California','New York'] errenkadak
df2 = df.copy()
df2.drop(['California','New York'], axis=0, inplace=True)
df2

## Datuekin eragiketak egiten
   * Eragile aritmetikoak eta NumPy funtzioak Series/DataFrame-ekin erabil daitezke
   * Indizeak mantendu egiten dira
   * Eragiketa/funtzio bitarrek indizeak lerrokatzen dituzte
      * Datu osatugabeak kudea ditzakete `NaN` balioak txertatuz
   * DataFrame eta Series arteko eragiketek broadcasting propietatea dute

### Indizeen mantentzea

In [ ]:
df

In [ ]:
df * 10

In [ ]:
np.log(df[['area','population']])

Python-eko eragileek metodo baliokideak dituzte Pandas objektuetan:

| Python-eko eragilea | Pandas metodoa(k)                |
|-----------------|---------------------------------|
| `+`             | `add`                           |
| `-`             | `sub`, `subtract`               |
| `*`             | `mul`, `multiply`               |
| `/`             | `truediv`, `div`, `divide`      |
| `//`            | `floordiv`                      |
| `%`             | `mod`                           |
| `**`            | `pow`                           |

### Indizeen lerrokatzea

In [ ]:
x = pd.Series({'A': 10,'B': 20})
y = pd.Series({'B': 4, 'A': 5})
x / y 

In [ ]:
x = pd.Series({'A': 10,'B': 20, 'C':30})
y = pd.Series({'B': 4, 'C': 5, 'D':3})
x / y 

In [ ]:
df1 = pd.DataFrame({
    'first':pd.Series([1,2],index=["A","B"]),
    'second':pd.Series([1,2],index=["A","B"])})
df2 = pd.DataFrame({
    'first':pd.Series([3,2],index=["A","C"]),
    'second':pd.Series([2,4],index=["A","C"])})

In [ ]:
display("df1", "df2" , "df1 + df2")

Pandas-en funtzio aritmetikoek `fill_value` parametroa dute falta diren sarreren ordez erabiltzeko:

In [ ]:
df1.add(df2)

In [ ]:
df1.add(df2, fill_value=0)

In [ ]:
df1.loc['C']=0
df2.loc['B']=0
display("df1" , "df2" , "df1+df2")

### DataFrame-en eta Series-en arteko eragiketak
* Broadcasting-a, NumPy-ren *antzekoa*
   * Errenkada-eragiketetara mugatua

In [ ]:
ints = np.random.randint(0,10,(3,4))
df = pd.DataFrame(ints, columns=['A','B','C','D'], index=['x','y','z'])
df

In [ ]:
display("df" , "df - df.loc['x']" , "df.sub(df.loc['x'], axis=1)")

In [ ]:
df

In [ ]:
# Broadcasting-aren arauak...
display("df - df['A']" , "df.sub(df['A'], axis=0)")

`pd.DataFrame(df['A'])` zutabe bakarreko DataFrame bat da, baina ez dago broadcasting-ik DataFrame-en artean...

In [ ]:
display("pd.DataFrame(df['A'])" , "df - pd.DataFrame(df['A'])")

## Falta diren datuen kudeaketa
Pandas-ek *balio jagoleak* (*Sentinel Values*) erabiltzen ditu erabilgarri ez dauden (NA) balioak kudeatzeko:
* `None`, `np.nan` and `pd.NA`
    * Zenbaki osoak: `pd.NA`
    * Zenbaki errealak: `np.nan` and `pd.NA`
    * Objektu orokorrak: `None`, `np.nan` and `pd.NA`
* Defektuz, zenbakizko datuetan, `None` &rarr; `np.nan` (floating point) 
<br>

In [ ]:
s = pd.Series([None,2,3])
pprint('s', 's[0]', 'type(s[0])', align=True)

In [ ]:
s = pd.Series([np.nan,2,3])
pprint('s', 's[0]', 'type(s[0])', align=True)

In [ ]:
s = pd.Series([pd.NA,2,3])
pprint('s', 's[0]', 'type(s[0])', align=True)

In [ ]:
s = pd.Series([None,2,3], dtype='Int64')
pprint('s', 's[0]', 'type(s[0])', align=True)

In [ ]:
s = pd.Series([np.nan,2,3], dtype='Int64')
pprint('s', 's[0]', 'type(s[0])', align=True)

In [ ]:
pprint('pd.Series([None,"a",123])')

In [ ]:
pprint('pd.Series([np.nan,"a",123])')

In [ ]:
pprint('pd.Series([pd.NA,"a",123])')

Pandas-en goranzko mota-bihurketaren konbentzioak, NA balioak sartzen direnean:

|Mota-klasea     | NA gordetzean egindako bihurketa | NA balio jagolea      |
|--------------|-----------------------------|------------------------|
| ``floating`` | Aldaketarik ez                   | ``np.nan``             |
| ``object``   | Aldaketarik ez                   | ``None`` edo ``np.nan`` |
| ``integer``  | ``float64`` bihurtu        | ``np.nan``             |
| ``boolean``  | ``object``  bihurtu        | ``None`` edo ``np.nan`` |

### Balio nuluak baztertzen
* NumPy-ko funtzio gehienek ez dituzte `nan` balioak kontuan hartzen (`nan` sortzen dute)
   * `nan` balioak kontuan hartzen dituzten NumPy funtzioak (`nan` balioak baztertzen dituzte):
      * `np.nansum`
      * `np.nanmin`
      * `np.nanmax`
   * NumPy-k ez du `pd.NA` kudeatzen &rarr; **ERROR** 
* Pandas funtzioek `NaN` eta `NA` kontuan hartzen dituzte (`nan` balioak baztertzen dituzte)

<br>

In [ ]:
s = pd.Series([1,2,None,3,4])
pprint('s.values',
       's.values.sum()', 's.values.min()', 's.values.max()',
       'np.sum(s.values)', 'np.min(s.values)', 'np.max(s.values)',
       'np.nansum(s.values)', 'np.nanmin(s.values)', 'np.nanmax(s.values)',
       's.sum()', 's.min()', 's.max()', align=True)

In [ ]:
s = pd.Series([1,2,pd.NA,3,4])
pprint('s.values',
       's.sum()', 's.min()', 's.max()', align=True)
# ERROREA!
#pprint('s.values.sum()', 's.values.min()', 's.values.max()',
#       'np.sum(s.values)', 'np.min(s.values)', 'np.max(s.values)',
#       'np.nansum(s.values)', 'np.nanmin(s.values)', 'np.nanmax(s.values)')

### Balio nuluak detektatzen

* `isnull` &rarr; (`True`/`False`) maskara boolearra balio nuluak dauden tokietan
* `notnull` &rarr; `isnull`-en aurkakoa 

<br>

In [ ]:
s = pd.Series([1,2,None,3,4])
pprint('s.isnull()', 's.notnull()', sep="\n\n", align=True)

In [ ]:
s[s.notnull()]

### Balio nuluak kentzen

* `dropna` &rarr; balio nuluak kenduta dituen kopia
   * Series: balio nuluak kendu
   * DataFrame: balio nuluak dituzten errenkadak/zutabeak kendu
* `inplace` parametroa: bool, defektuz `False`
   * `True` bada, lekuan bertan kentzen/betetzen du. Oharra: objektu honen beste bistak ere aldatuko ditu (adibidez, DataFrame bateko zutabe baten kopiarik gabeko zatia).

<br>

In [ ]:
s = pd.Series([1,2,None,3,4])
s.dropna() 

In [ ]:
pprint('s',end='\n\n')
s.dropna(inplace=True) 
pprint('s')

In [ ]:
a = np.arange(15,dtype='float64').reshape(3,5)
a[(0,2,2),(3,0,2)] = np.nan
df = pd.DataFrame(a)
df

In [ ]:
df.dropna() # df.dropna(axis=0)

In [ ]:
df.dropna(axis=1)

### Balio nuluak betetzen

* `fillna` &rarr; balio nuluak ordezkatuta dituen kopia
* `inplace` parametroa: bool, lehenetsia False
   * `True` bada, lekuan bertan kentzen/betetzen du. Oharra: objektu honen beste bistak ere aldatuko ditu (adibidez, DataFrame bateko zutabe baten kopiarik gabeko zatia).
* **ZAHARKITUTAKO** `method` parametroa: {`bfill`, `ffill`, `None`}, defektuz `None`
   * `ffill` &rarr; *aurrerantz betetzea*,  aurreko balioa aurrerantz hedatu
   * `bfill` &rarr; *atzerantz betetzea*,  hurrengo balioa atzerantz hedatu
* `data.ffill()` eta `data.bfill()` erabili honen ordez:
   * `axis` parametroa: aurrerantz/atzerantz betetzeko ardatza

<br>

In [ ]:
s = pd.Series([1, None, 2, None, 3], index=list('abcde'), dtype='Int64')
pprint('s', 's.fillna(0)', 's.ffill()', 's.bfill()', align=True)

In [ ]:
a = np.arange(15,dtype='float64').reshape(3,5)
a[(0,2,2),(3,0,2)] = np.nan
df = pd.DataFrame(a)

display("df" , "df.fillna(0)")

In [ ]:
df

In [ ]:
display("df.ffill()" , "df.ffill(axis=1)")

In [ ]:
df

In [ ]:
display("df.bfill()" , "df.bfill(axis=1)")

## Datu multzoak kateatzen

In [ ]:
def make_df(col_index,row_index):
    """DataFrame bat azkar sortu"""
    data = {c: [str(c) + str(r) for r in row_index] for c in col_index}
    return pd.DataFrame(data, index=list(row_index), columns=list(col_index))

# DataFrame adibidea
display("make_df('ABCDEF',range(3))" , "make_df('ABCDEF','xyz')")

### `pd.concat()`
* NumPy-ko `concatenate`-ren antzekoa

```python
pd.concat(objs, axis=0, join='outer', ignore_index=False, keys=None,
          levels=None, names=None, verify_integrity=False,
          sort=False, copy=True)
```

In [ ]:
s1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
s2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
pd.concat([s1, s2])

In [ ]:
s1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
s2 = pd.Series(['D', 'E', 'F'], index=[1, 2, 3])
pd.concat([s1, s2], axis=1)

In [ ]:
df1 = make_df('AB', 'uv')
df2 = make_df('AB', 'xy')
display('df1', 'df2', 'pd.concat([df1, df2])')

In [ ]:
df1 = make_df('AB', 'xy')
df2 = make_df('CD', 'xy')
display('df1', 'df2', 'pd.concat([df1, df2], axis=1)')

* **Indizeen mantentzea**: `pd.concat`-ek indizea *mantentzen* du (emaitzak indize bikoiztuak izan ditzake)

In [ ]:
df1 = make_df('AB', 'xy')
df2 = make_df('AB', 'xy')
df_axis0 = pd.concat([df1, df2])
df_axis1 = pd.concat([df1, df2], axis=1)
display('df1', 'df2' , 'df_axis0' , 'df_axis1')

* `verify_integrity=True` indizeak bikoiztea eragozten du ( `ValueError`)

In [ ]:
df1 = make_df('AB', 'xy')
df2 = make_df('AB', 'xy')
try:
    pd.concat([df1, df2], verify_integrity=True)
except ValueError as e:
    print("axis=0 ValueError:", e)
try:
    pd.concat([df1, df2], axis=1, verify_integrity=True)
except ValueError as e:
    print("axis=1 ValueError:", e)
display('df1', 'df2')

* `ignore_index=True` gainjarritako indizeak baztertzen ditu (`RangeIndex`-ekin ordezkatzen dira)

In [ ]:
df1 = make_df('AB', 'xy')
df2 = make_df('AB', 'xy')
df_axis0 = pd.concat([df1, df2], ignore_index=True)
df_axis1 = pd.concat([df1, df2], axis=1, ignore_index=True)
pprint('df_axis0.index', 'df_axis0.columns',
       'df_axis1.index', 'df_axis1.columns', align=True)
display('df1', 'df2' , 'df_axis0' , 'df_axis1')

* `join="outer"` (defektuz): datu nuluak sor ditzake
* `join="inner`: datu nuluak saihesten ditu
   * `axis=0` &rarr; zutabeak kentzen ditu
   * `axis=1` &rarr; errenkadak kentzen ditu

In [ ]:
df1 = make_df('ABC', [1,2])
df2 = make_df('BCD', [3,4])
display('df1', 'df2')

In [ ]:
df_outer = pd.concat([df1, df2])
df_inner = pd.concat([df1, df2], join='inner')
display('df_outer', 'df_inner')

In [ ]:
df1 = make_df('AB', [1,2,3])
df2 = make_df('CD', [2,3,4])
display('df1', 'df2')

In [ ]:
df_outer = pd.concat([df1, df2], axis=1)
df_inner = pd.concat([df1, df2], axis=1, join='inner')
display('df_outer', 'df_inner')

## Dataset-ak bateratzen

### `pd.merge()`

* Bi Series/DataFrame bateratzen ditu
* Zutabe *komunak* bateratze-gako gisa erabiltzen dira

```python
pd.merge(left, right, how='inner', on=None, left_on=None, right_on=None,
         left_index=False, right_index=False, sort=False,
         suffixes=('_x', '_y'), copy=None, indicator=False, validate=None)
```

In [ ]:
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa'],
                    'group': ['Accounting', 'Engineering','Engineering']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake'],
                    'hire_date': [2004, 2008, 2012]})
display('df1', 'df2')

In [ ]:
pd.merge(df1, df2)

In [ ]:
df1 = pd.merge(df1, df2)
df2 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})
display('df1', 'df2')

In [ ]:
pd.merge(df1, df2)

In [ ]:
df2 = pd.DataFrame({'group': ['Accounting', 'Accounting', 'Engineering', 'Engineering'],
                    'skills': ['math', 'spreadsheets', 'software', 'math']})
display('df1', 'df2')

In [ ]:
pd.merge(df1, df2)

* `on=column_name` &rarr; zutabea gako gisa erabili
* `left_on=name1`, `right_on=name2` &rarr; zutabeak gako gisa erabili

In [ ]:
df2 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'], 'salary': [70000, 80000, 120000, 90000]})
display('df1', 'df2')

In [ ]:
pd.merge(df1, df2, left_on="employee", right_on="name")

`left_on=name1`, `right_on=name2` erabiltzeak zutabe erredundante bat sortzen du

In [ ]:
pd.merge(df1, df2, left_on="employee", right_on="name")

In [ ]:
pd.merge(df1, df2, left_on="employee", right_on="name").drop('name', axis=1)

* `left_index=True`, `right_index=True` &rarr; indizeak gako gisa erabili

In [ ]:
df2 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa'],
                    'salary': [70000, 80000, 120000]})
df2.set_index('name', inplace=True)
display('df1', 'df2')

In [ ]:
pd.merge(df1, df2, left_on='employee', right_index=True)

* `df1.join(df2)` &rarr; indizeetan oinarritutako bateratzea
   * `pd.merge(df1, df2, left_index=True, right_index=True)`

In [ ]:
df1.set_index('employee', inplace=True)
display('df1', 'df2')

In [ ]:
df1.join(df2)

* `how='inner'` (lehenetsia) &rarr; ebakidura

In [ ]:
df1 = pd.DataFrame({'name': ['Peter', 'Paul', 'Mary', 'Adam'],
                    'food': ['fish', 'beans', 'bread' , 'fish']})
df2 = pd.DataFrame({'name': ['Mary', 'Joseph', 'Peter' , 'Alice'],
                    'drink': ['wine', 'beer' , 'water' , 'beer']})
display('df1', 'df2', 'pd.merge(df1, df2)')

* `how='outer'` &rarr; bilketa (balio nuluak)

In [ ]:
display('df1', 'df2', 'pd.merge(df1, df2, how="outer")')

* `how='left'` &rarr; mantendu ezkerreko gakoa (balio nuluak)

In [ ]:
display('df1', 'df2', 'pd.merge(df1, df2, how="left")')

* `how='right'` &rarr; mantendu eskuineko gakoa (balio nuluak)

In [ ]:
display('df1', 'df2', 'pd.merge(df1, df2, how="right")')

## Agregazioa eta taldekatzea

### Planeten datuak

Astronomoek aurkitutako exoplanetei buruzko informazioa (eguzkia ez den beste izar baten inguruan orbitatzen duten planetak).

In [ ]:
import seaborn as sns
planets = sns.load_dataset('planets')
planets.shape

In [ ]:
planets.head(10)  #planets[:10]

In [ ]:
planets.dropna(inplace=True)
print(planets.shape)
planets.head(10)

### Agregazio-funtzioak

Pandas-en berezko (*built-in*) agregazio batzuk:

| Agregazioa              | Deskribapena                     |
|--------------------------|---------------------------------|
| ``count()``              | Elementu kopurua           |
| ``first()``, ``last()``  | Lehen eta azken elementua             |
| ``mean()``, ``median()`` | Batezbestekoa eta mediana                 |
| ``min()``, ``max()``     | Minimoa eta maximoa             |
| ``std()``, ``var()``     | Desbideratze estandarra eta bariantza |
| ``mad()``                | Batez besteko desbideratze absolutua         |
| ``sum()`` , ``prod()``   | Elementu guztien batura eta biderkadura    |

In [ ]:
planets = sns.load_dataset('planets')
df = planets.dropna().drop(['method','number'], axis=1)
print(df.shape)
pprint('df.min()', 'df.max()', 'df.mean()', 'df.std()', align=True)

In [ ]:
df = planets.drop(['method','number'], axis=1)
print(df.shape)
pprint('df.min()', 'df.max()', 'df.mean()', 'df.std()')

### GroupBy
* `df.groupby()` &rarr; DataFrame *bilduma* bat
   * DataFrame baten bista berezi moduko bat
* DataFrame bat bezala indexa daiteke &rarr; GroupBy objektu berri bat
* *zatitu-aplikatu-konbinatu* motako eragiketa errazak eskaintzen ditu

<center><img src="img/split-apply-combine.png" alt="split-apply-combine" style="width: 80%;"/></center>

In [ ]:
planets.groupby('method')

In [ ]:
planets.groupby('method')['orbital_period']

In [ ]:
planets.groupby('method')['orbital_period'].median()

GroupBy objektuak esplizituki inplementatzen ez dituen metodoak taldeetara pasatu eta haietan ebaluatzen dira.

In [ ]:
planets['orbital_period'].describe()

In [ ]:
planets.groupby('method')['orbital_period'].describe()

GroupBy objektua iteragarria da
* `(groupby_value , Series/DataFrame)` bikote sekuentzia bat 

In [ ]:
for method,df in planets.groupby('method'):
    print(f'{method:30s} {df.shape=}')

### Agregatu, iragazi, transformatu eta aplikatu
GroupBy objektuek `aggregate()`, `filter()`, `transform()`, eta `apply()` metodoak dituzte; metodo horiek hainbat eragiketa erabilgarri modu eraginkorrean inplementatzen dituzte taldekatutako datuak konbinatu aurretik.


In [ ]:
planets.groupby('method')[['orbital_period','distance']].aggregate(['count','min','max'])

## Denbora-serieekin lan egiten

* Pandas datekin, instanteekin eta denboraren arabera indexatutako datuekin bateragarria da
* Denboran oinarritutako indize bat erabil daiteke denbora-serietarako
* Indexazio estandarra eta datei dagokien indexazio berezia

### Python-eko datak eta instanteak: `datetime` and `dateutil`

In [ ]:
from datetime import datetime
date = datetime(year=2025, month=4, day=14)
date

<br>

`dateutil` moduluak hainbat formatutako karaktere-kateak ulertzen ditu:

In [ ]:
from dateutil import parser
date = parser.parse("14th of April, 2025")
date

### NumPy-ren `datetime64`

In [ ]:
import numpy as np
date = np.array('2025-04-14', dtype=np.datetime64)
date

<br>

Eragiketa bektorizatuak datekin erabil daitezke:

In [ ]:
date + np.arange(20)

### Datak eta instanteak Pandas-en

In [ ]:
date = pd.to_datetime("4th of July, 2015")
date

<br>

Eragiketa bektorizatuak datekin erabil daitezke:

In [ ]:
date + pd.to_timedelta(np.arange(20), 'D')

### Datak eta instanteak Pandas-en
`DatetimeIndex`-ek denbora-indize bat eskaintzen du

In [ ]:
index = pd.DatetimeIndex(['2014-07-04', '2014-08-04',
                          '2015-07-04', '2015-08-04'])
data = pd.Series([0, 1, 2, 3], index=index)
data

Indexazio-mota estandarrak:

In [ ]:
pprint("data['2014-08-04']", "data[:'2015-07-04']", "data[['2014-07-04','2015-07-04']]", align=True, sep="\n\n")

Datetan oinarritutako indexazioak:

In [ ]:
pprint("data['2014']", "data[:'2015-07']", align=True, sep="\n\n")


### Denbora-serieen datu-egiturak
* *denbora-zigiluak*: `Timestamp` (np.datetime64) &rarr; `DatetimeIndex`
* *denbora-tarteak*: `Period` (np.datetime64) &rarr; `PeriodIndex`
* *denbora-diferentziak edo iraupenak*: `Timedelta` (np.timedelta64) &rarr; `TimedeltaIndex`

<br>

* `pd.to_datetime()` (analizatzailea, *parser*) &rarr; `Timestamp` or `DatetimeIndex`

In [ ]:
dates = pd.to_datetime([datetime(2015, 7, 3), '4th of July, 2015',
                       '2015-Jul-6', '07-07-2015', '20150708'])
pprint('dates','dates[0]', align=True)

* `DatetimeIndex.to_period()` &rarr; `PeriodIndex`

In [ ]:
dates.to_period('D')

In [ ]:
days = dates.to_period('D')
months = dates.to_period('M')
years = dates.to_period('Y')
pprint('dates[:2]', 'days[:2]', 'months[:2]', 'years[:2]', align=True)

* `Timestamp - Timestamp` &rarr; `Timedelta`:

In [ ]:
dates[1]-dates[0]

* `DatetimeIndex - Timestamp` &rarr; `TimedeltaIndex `:

In [ ]:
dates-dates[0]

* `pd.date_range()` &rarr; `DatetimeIndex`
* `pd.period_range()` &rarr; `PeriodIndex`
* `pd.timedelta_range()` &rarr; `TimedeltaIndex`

In [ ]:
pd.date_range('2015-07-03', '2015-07-10')

In [ ]:
pd.date_range('2015-07-03', periods=8, freq='h')

In [ ]:
pd.period_range('2015-07', periods=8, freq='M')

In [ ]:
pd.timedelta_range(0, periods=10, freq='h')

### Ber-lagintzea (*Resampling*), desplazatzea (*Shifting*) eta leihokatzea (*Windowing*)

In [ ]:
#!pip install pandas-datareader
from pandas_datareader import data
#goog = data.DataReader('GOOG', start=2020, end='2024',data_source='stooq')
goog = data.DataReader('GOOG', start=2020, end='2024', data_source='stooq').sort_index()
goog.head()

onfigurazio bisual batzuk...

In [ ]:
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn
seaborn.set()
mpl.rcParams['figure.figsize'] = (5.33,4)
mpl.rcParams['axes.labelsize'] = 10  # Adibidea: 14 puntu
mpl.rcParams['xtick.labelsize'] = 8  # Adibidea: 12 puntu x ardatzeko marketarako
mpl.rcParams['ytick.labelsize'] = 8  # Adibidea: 12 puntu y ardatzeko marketarako

In [ ]:
goog = goog['Close']

In [ ]:
goog.plot(linewidth=0.5);

* `resample(freq)` &rarr; agregazio/murrizketa eragiketa baterako prest dagoen objektu taldekatua
* `resample(freq).aggregate_func()` &rarr; agregatutako balioak dituen denbora-serie berria
* `asfreq(freq)` &rarr; hautatutako/aldizkako balioak dituen denbora-serie berria
   * $\approx$ `resample(freq).last()`

In [ ]:
# 'BYE': Negozio-urtearen amaiera (urteko azken laneguna)
goog.plot(alpha=0.5, style='-', linewidth=0.5)
goog.resample('BYE').mean().plot(style=':') 
goog.asfreq('BYE').plot(style='--');
plt.legend(['original', 'resample', 'asfreq'], loc='upper left');

Berriz lagintzeak (`resample+aggregation` edo `asfreq`) balio nuluak sor ditzake
* `method="ffill"|"bfill"` &rarr; balioak nola egozten diren

In [ ]:
d = goog.head()
pprint("d" , "d.asfreq('D')", "d.asfreq('D',method='ffill')", align=True)

In [ ]:
fig, ax = plt.subplots(2, sharex=True)
data = goog.head()

data.asfreq('D').plot(ax=ax[0], marker='o')

data.asfreq('D', method='bfill').plot(ax=ax[1], style='-o')
data.asfreq('D', method='ffill').plot(ax=ax[1], style='--o')
ax[1].legend(["back-fill", "forward-fill"]);

* `shift()` &rarr; datuak desplazatu
   *  `shift(int)` &rarr; datuak `int` posizio desplazatu
   *  `shift(int, freq)` &rarr; datuak `freq`-eko `int` kopurua desplazatu

In [ ]:
pprint("goog.head()", "goog.shift(3).head()", "goog.shift(3,'D').head()", align=True)

Kalkulatu denboran zeharreko diferentziak (maiztasunak):

In [ ]:
pprint("goog.head()", "goog.shift(-365,'D').head()", align=True)

In [ ]:
data = goog.asfreq('D', method='ffill')
ROI = 100 * (data.shift(-365,'D') / data - 1)
ROI.plot()
plt.ylabel('% Return on Investment');

* `rolling()` &rarr; leiho mugikorra
   *  agregazio/murrizketa funtzioak leiho mugikor bati aplika dakizkioke

In [ ]:
r365 = goog.rolling(365).mean()
r200 = goog.rolling(200).mean()
data = pd.DataFrame({'GOOG': goog, '1year mean': r365, '200d mean': r200})
ax = data.plot(style=['-', '--', ':'])
ax.lines[0].set_alpha(0.3)


`rolling(365)` ez da urtebeteko leiho mugikorra...

In [ ]:
r365 = goog.asfreq('D', method='ffill').rolling(365).mean()
r200 = goog.asfreq('D', method='ffill').rolling(200).mean()
data = pd.DataFrame({'GOOG': goog, '1year mean': r365, '200d mean': r200})
ax = data.plot(style=['-', '--', ':'])
ax.lines[0].set_alpha(0.3)